In [0]:
# Create widgets
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

#setting up some more variables
landing_volume = "/Volumes/adb_qnxthealth/landing/landing_volume"

#setting source file locations
landing_source_path = f"{landing_volume}/{source_system}/{table_name}/{landing_timestamp}/"

#print(f"source_system: {source_system}")
#print(f"table_name: {table_name}")
#print(f"landing_timestamp: {landing_timestamp}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_source_path)
else:
    landing_df = spark.read.format("parquet").option("inferSchema", "true").load(landing_source_path)
#display(landing_df.limit(5))

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_landing_df = landing_df\
        .withColumn("landing_timestamp", lit(landing_timestamp)) \
        .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_landing_df.write\
        .mode("append")\
        .saveAsTable(f"adb_qnxthealth.bronze.{table_name}")